In [1]:
import numpy as np
import pandas as pd
from scipy import stats

# 1. Select metric type here ("std" or "ci")
STAT_TYPE = "std"  # Change to "ci" whenever you want Confidence Intervals


def summarize_metric(group, column, stat_type=STAT_TYPE, confidence=0.95):
    """Summarizes a column as Mean ± [Std | CI] (raw_values)."""
    values = group[column].dropna().to_numpy(dtype=float)
    n = len(values)

    if n == 0:
        return "N/A"

    mean = np.mean(values)

    if n > 1:
        if stat_type == "std":
            margin = np.std(values, ddof=1)
        elif stat_type == "ci":
            sem = stats.sem(values)
            margin = sem * stats.t.ppf((1 + confidence) / 2, df=n - 1)
        else:
            raise ValueError("stat_type must be either 'std' or 'ci'")

        margin_str = f"{margin:.2f}"
    else:
        margin_str = "N/A"

    actual = ", ".join(f"{x:.2f}" for x in values)

    return f"{mean:.2f} ± {margin_str} ({actual})"




from pathlib import Path
import json

import numpy as np
import pandas as pd


def summarize_run(run_dir, tail_n=5):
    run_dir = Path(run_dir)

    with open(run_dir / "args.json", encoding="utf-8") as f:
        args = json.load(f)

    fid_path = run_dir / "validation_fid.jsonl"
    fids = []

    if fid_path.exists():
        with open(fid_path, encoding="utf-8") as f:
            fids = [
                json.loads(line)
                for line in f
                if line.strip()
            ]

    row = {
        "run": run_dir.name,
        **{f"arg_{key}": value for key, value in args.items()},
        "fid_best": np.nan,
        "fid_best_iteration": np.nan,
        "fid_final": np.nan,
        "fid_final_iteration": np.nan,
        "fid_tail_median": np.nan,
        "fid_tail_mean": np.nan,
        "fid_tail_max": np.nan,
        "fid_tail_n": 0,
    }

    if fids:
        values = np.array([item["fid"] for item in fids], dtype=float)
        iterations = np.array([item["iteration"] for item in fids])

        tail = values[-tail_n:]

        best_index = np.argmin(values)

        row.update({
            "fid_best": values[best_index],
            "fid_best_iteration": iterations[best_index],
            "fid_final": values[-1],
            "fid_final_iteration": iterations[-1],
            "fid_tail_median": np.median(tail),
            "fid_tail_mean": np.mean(tail),
            "fid_tail_max": np.max(tail),
            "fid_tail_n": len(tail),
        })

    return row

def summarize_runs(experiment_dirs, tail_n=5):
    """Summarizes all run subdirectories found inside multiple experiment directories."""
    rows = []

    # Handle both a single path string/Path object and a list of paths
    if isinstance(experiment_dirs, (str, Path)):
        experiment_dirs = [experiment_dirs]

    for exp_dir in experiment_dirs:
        exp_path = Path(exp_dir)

        # Iterate through subdirectories inside each experiment root
        for run_dir in exp_path.iterdir():
            # Check if it's a directory and contains args.json
            if run_dir.is_dir() and (run_dir / "args.json").exists():
                rows.append(summarize_run(run_dir, tail_n=tail_n))

    return pd.DataFrame(rows)

# Example usage:


In [2]:
# 2. Build comparison DataFrame
suffix = "std" if STAT_TYPE == "std" else "95ci"

dirs = [
    "/home/satoshi/projects/fcmstylegan/experiments/eeeg/dcgan/explore_ema_diffaug/",
    "/home/satoshi/projects/fcmstylegan/experiments/eeeg/dcgan/explore_profencoder/",
]
df = summarize_runs(dirs)


comparison = (
    df.groupby(
        [
            # "arg_seed",
            "fid_final_iteration",
            "arg_base_channels",
            "arg_profile_encoder",
            "arg_diff_aug_policy",
            "arg_ema",
        ],
        dropna=False,
    )
    .apply(
        lambda g: pd.Series({
            "runs": len(g),
            f"tail_fid_{suffix}": summarize_metric(g, "fid_tail_median", stat_type=STAT_TYPE),
            f"best_fid_{suffix}": summarize_metric(g, "fid_best", stat_type=STAT_TYPE),
            f"final_fid_{suffix}": summarize_metric(g, "fid_final", stat_type=STAT_TYPE),
            f"tail_max_fid_{suffix}": summarize_metric(g, "fid_tail_max", stat_type=STAT_TYPE),
        })
    )
)

display(comparison)

runs  \
fid_final_iteration arg_base_channels arg_profile_encoder arg_diff_aug_policy arg_ema         
200000              512               cnn                                     False       7   
                                      mlp                                     False       2   
                                                                              True        2   
                                                          color               False       2   
                                                                              True        2   
                                                          cutout              False       2   
                                                                              True        2   
                                                          translation         False       2   
                                                                              True        2   

                                                                                                                            tail_fid_std  \
fid_final_iteration arg_base_channels arg_profile_encoder arg_diff_aug_policy arg_ema                                                      
200000              512               cnn                                     False    64.51 ± 16.34 (58.82, 57.03, 101.46, 57.77, 57...   
                                      mlp                                     False                          41.54 ± 4.38 (44.64, 38.44)   
                                                                              True                           40.23 ± 2.30 (38.60, 41.86)   
                                                          color               False                         47.89 ± 16.27 (59.40, 36.38)   
                                                                              True                           41.83 ± 3.74 (44.47, 39.18)   
                                                          cutout              False                       122.12 ± 86.70 (183.43, 60.81)   
                                                                              True                       152.60 ± 154.18 (43.58, 261.62)   
                                                          translation         False                         76.51 ± 26.39 (95.16, 57.85)   
                                                                              True                         100.32 ± 8.46 (94.33, 106.30)   

                                                                                                                            best_fid_std  \
fid_final_iteration arg_base_channels arg_profile_encoder arg_diff_aug_policy arg_ema                                                      
200000              512               cnn                                     False    39.09 ± 5.28 (39.50, 45.95, 34.52, 34.32, 34.3...   
                                      mlp                                     False                          27.18 ± 1.80 (28.46, 25.91)   
                                                                              True                           32.58 ± 1.25 (33.47, 31.70)   
                                                          color               False                          32.66 ± 5.63 (36.64, 28.68)   
                                                                              True                           34.68 ± 1.14 (33.87, 35.48)   
                                                          cutout              False                          31.41 ± 1.32 (32.34, 30.48)   
                                                                              True                           37.27 ± 2.98 (35.17, 39.38)   
                                                          translation         False                          32.19 ± 6.44 (36.74, 27.64)   
                                                                              True                  

### Takeaways
- EMA does make it slightly worse. 
- diff_aug does not really help, at least for this settings. making lr lower or differnet base channels may make a difference. 
- somehow cnn profile encoder seems to do worse...